In [59]:
# === NOTEBOOK 02: CLUSTERING + RECOMMENDATION v3 ===
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count, desc
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
import pandas as pd
import folium
from IPython.display import display

spark = SparkSession.builder.getOrCreate()

print("="*70)
print(" NOTEBOOK 02: CLUSTERING + RECOMMENDATION ENGINE v3")
print("="*70)

# Load data
df = spark.read.parquet("/lakehouse/default/Files/umkm_v3_final")

print(f"\n✅ Data loaded: {df.count()} UMKM")

# 1. K-Means Clustering (8 cluster)
print("\n>>> Melakukan K-Means Clustering (8 cluster)...")

assembler = VectorAssembler(inputCols=["latitude", "longitude"], outputCol="features")
df_features = assembler.transform(df)

kmeans = KMeans(k=8, seed=42, featuresCol="features", predictionCol="cluster_new")
model = kmeans.fit(df_features)
df = model.transform(df_features).drop("features", "cluster").withColumnRenamed("cluster_new", "cluster")

print("✅ Clustering selesai")

# 2. Top 15 Rekomendasi per Jenis Usaha
print("\n" + "="*70)
print(" TOP 15 REKOMENDASI LOKASI TERBAIK")
print("="*70)

jenis_list = ["Makanan", "Fashion", "Kerajinan", "Jasa", "Pertanian"]

for jenis in jenis_list:
    print(f"\n>>> {jenis.upper()}")
    print("-"*50)
    
    rec = df.filter(col("jenis_usaha") == jenis) \
            .orderBy(col("skor_potensi").desc()) \
            .select("umkm_id", "kabupaten", "kecamatan", "skor_potensi", 
                    "persaingan_index", "risiko_banjir", "cluster") \
            .limit(15)
    
    rec.show(15, truncate=False)

# 3. What-If Analysis
print("\n" + "="*70)
print(" WHAT-IF ANALYSIS (5 Contoh Lokasi)")
print("="*70)

sample_locs = df.select("kabupaten", "kecamatan", "jenis_usaha").distinct().limit(5).collect()

for loc in sample_locs:
    sample = df.filter(
        (col("kabupaten") == loc["kabupaten"]) & 
        (col("kecamatan") == loc["kecamatan"]) &
        (col("jenis_usaha") == loc["jenis_usaha"])
    ).limit(1).collect()[0]
    
    print(f"\n📍 {loc['jenis_usaha']} di {loc['kecamatan']}, {loc['kabupaten']}")
    print(f"   Skor Potensi    : {sample['skor_potensi']:.1f}/100")
    print(f"   Persaingan      : {sample['persaingan_index']:.2f}")
    print(f"   Risiko Banjir   : {sample['risiko_banjir']:.2f}")
    print(f"   Cluster         : {sample['cluster']}")
    
    if sample['persaingan_index'] > 0.7:
        print("   💡 Saran: Pertimbangkan diferensiasi produk")

# 4. Government Priority (15 Kecamatan Skor Terendah)
print("\n" + "="*70)
print(" 15 KECAMATAN DENGAN SKOR TERENDAH (PRIORITAS PEMERINTAH)")
print("="*70)

lowest = df.groupBy("kabupaten", "kecamatan") \
    .agg(
        avg("skor_potensi").alias("avg_score"),
        count("*").alias("jumlah_umkm"),
        avg("persaingan_index").alias("avg_competition")
    ) \
    .orderBy(col("avg_score").asc()) \
    .limit(15)

lowest.select("kabupaten", "kecamatan", "avg_score", "jumlah_umkm", "avg_competition").show(15, truncate=False)

print("\n💡 REKOMENDASI: Fokuskan program di 15 kecamatan ini")

# 5. Peta Interaktif
print("\n" + "="*70)
print(" PETA INTERAKTIF (Skor 45-95)")
print("="*70)

pdf = df.select("latitude", "longitude", "skor_potensi", "jenis_usaha", "kecamatan").limit(1500).toPandas()

m = folium.Map(location=[-6.9, 107.6], zoom_start=9, tiles="OpenStreetMap")

for _, row in pdf.iterrows():
    color = 'green' if row['skor_potensi'] >= 75 else \
            'orange' if row['skor_potensi'] >= 60 else 'red'
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.5,
        popup=f"Skor: {row['skor_potensi']:.1f}<br>{row['jenis_usaha']}<br>{row['kecamatan']}"
    ).add_to(m)

legend = '''
<div style="position:fixed; bottom:50px; left:50px; width:200px; background:white; 
            border:2px solid grey; padding:10px; font-size:11px; z-index:9999;">
<b>LEGENDA - GeoUMKM v3</b><br>
🟢 Skor Tinggi (≥75)<br>
🟠 Skor Sedang (60-75)<br>
🔴 Skor Rendah (<60)
</div>
'''
m.get_root().html.add_child(folium.Element(legend))

display(m)
print("\n✅ Peta berhasil ditampilkan!")

StatementMeta(yudhaesparkpool, 2, 66, Finished, Available, Finished, False)

 NOTEBOOK 02: CLUSTERING + RECOMMENDATION ENGINE v3

✅ Data loaded: 6000 UMKM

>>> Melakukan K-Means Clustering (8 cluster)...
✅ Clustering selesai

 TOP 15 REKOMENDASI LOKASI TERBAIK

>>> MAKANAN
--------------------------------------------------
+-------------+-----------+------------+------------+----------------+-------------+-------+
|umkm_id      |kabupaten  |kecamatan   |skor_potensi|persaingan_index|risiko_banjir|cluster|
+-------------+-----------+------------+------------+----------------+-------------+-------+
|JB-2026-01080|Purwakarta |Kecamatan 4 |90.0        |0.47            |0.32         |5      |
|JB-2026-04401|Purwakarta |Kecamatan 28|89.0        |0.45            |0.36         |7      |
|JB-2026-03649|Bandung    |Kecamatan 24|89.0        |0.42            |0.26         |5      |
|JB-2026-00307|Banjar     |Kecamatan 1 |88.0        |0.29            |0.31         |0      |
|JB-2026-01881|Karawang   |Kecamatan 12|88.0        |0.35            |0.45         |2      |
|JB-2026


✅ Peta berhasil ditampilkan!
